#  UK Retail Sales Analysis
**Author:** Kiruthik Raam D V  
**MSc Data Science** — University of Essex  
**Dataset:** [Online Retail Dataset — Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset)

---

## Project Overview
This project analyses two years of real UK online retail transaction data to uncover:
- Best-selling products and categories
- Monthly and seasonal sales trends
- Top customer countries
- Customer segmentation using RFM analysis
- Revenue forecasting

**Business Goal:** Help a UK retailer understand their sales patterns and make data-driven decisions.

## 1.  Install & Import Libraries

In [1]:
# Install required libraries
!pip install kaggle plotly nbformat -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ Libraries loaded successfully!')

✅ Libraries loaded successfully!


## 2.  Load the Dataset
> **Instructions:**
> 1. Go to [Kaggle Dataset](https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset)
> 2. Download `online_retail_II.xlsx`
> 3. Upload it to this Colab session using the cell below

In [2]:
# Upload the file to Colab
from google.colab import files
uploaded = files.upload()

Saving online_retail_II.xlsx to online_retail_II.xlsx


In [3]:
# Load both sheets (Year 1 and Year 2)
df1 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')

# Combine into one dataframe
df = pd.concat([df1, df2], ignore_index=True)

print(f'Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns')
df.head()

Dataset loaded: 1,067,371 rows and 8 columns


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 3. Data Exploration

In [4]:
# Basic info
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB
None

=== Missing Values ===
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

=== Basic Statistics ===


,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [5]:
# Check unique values
print(f'Unique Customers : {df["Customer ID"].nunique():,}')
print(f'Unique Products  : {df["StockCode"].nunique():,}')
print(f'Unique Countries : {df["Country"].nunique()}')
print(f'Date Range       : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')

Unique Customers : 5,942
Unique Products  : 5,305
Unique Countries : 43
Date Range       : 2009-12-01 07:45:00 → 2011-12-09 12:50:00


## 4.  Data Cleaning

In [6]:
print(f'Before cleaning: {df.shape[0]:,} rows')

# Drop rows with missing Customer ID or Description
df = df.dropna(subset=['Customer ID', 'Description'])

# Remove cancelled orders (Invoice starts with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove negative or zero quantities and prices
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['Price']

# Extract date features
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year']  = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Month_Name'] = df['InvoiceDate'].dt.strftime('%b')
df['Day']   = df['InvoiceDate'].dt.day_name()
df['Hour']  = df['InvoiceDate'].dt.hour

print(f'After cleaning : {df.shape[0]:,} rows')
print(f'Total Revenue  : £{df["Revenue"].sum():,.2f}')

Before cleaning: 1,067,371 rows
After cleaning : 805,549 rows
Total Revenue  : £17,743,429.18


## 5. Sales Trends Over Time

In [7]:
# Monthly revenue trend
monthly = df.groupby(df['InvoiceDate'].dt.to_period('M'))['Revenue'].sum().reset_index()
monthly['InvoiceDate'] = monthly['InvoiceDate'].astype(str)

fig = px.line(
    monthly,
    x='InvoiceDate',
    y='Revenue',
    title='Monthly Revenue Trend (2009–2011)',
    labels={'Revenue': 'Total Revenue (£)', 'InvoiceDate': 'Month'},
    markers=True
)
fig.update_traces(line_color='#2196F3', line_width=2.5)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print(f'Peak month: {monthly.loc[monthly["Revenue"].idxmax(), "InvoiceDate"]} — £{monthly["Revenue"].max():,.2f}')

Peak month: 2010-11 — £1,172,336.04


In [8]:
# Revenue by day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_revenue = df.groupby('Day')['Revenue'].sum().reindex(day_order).reset_index()

fig = px.bar(
    day_revenue,
    x='Day',
    y='Revenue',
    title=' Revenue by Day of Week',
    color='Revenue',
    color_continuous_scale='Blues'
)
fig.show()

In [9]:
# Revenue by hour
hour_revenue = df.groupby('Hour')['Revenue'].sum().reset_index()

fig = px.bar(
    hour_revenue,
    x='Hour',
    y='Revenue',
    title=' Revenue by Hour of Day',
    labels={'Hour': 'Hour of Day', 'Revenue': 'Total Revenue (£)'},
    color='Revenue',
    color_continuous_scale='Teal'
)
fig.show()

## 6. Top Products

In [10]:
# Top 10 products by revenue
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(
    top_products,
    x='Revenue',
    y='Description',
    orientation='h',
    title=' Top 10 Products by Revenue',
    color='Revenue',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [11]:
# Top 10 products by quantity sold
top_qty = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(
    top_qty,
    x='Quantity',
    y='Description',
    orientation='h',
    title='Top 10 Products by Quantity Sold',
    color='Quantity',
    color_continuous_scale='Oranges'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 7. Sales by Country

In [12]:
# Revenue by country (excluding UK to see export markets)
country_rev = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).reset_index()

# Top 10 countries
top_countries = country_rev.head(10)

fig = px.bar(
    top_countries,
    x='Country',
    y='Revenue',
    title=' Top 10 Countries by Revenue',
    color='Revenue',
    color_continuous_scale='Blues'
)
fig.show()

# World map
fig2 = px.choropleth(
    country_rev,
    locations='Country',
    locationmode='country names',
    color='Revenue',
    title=' Global Revenue Distribution',
    color_continuous_scale='Blues'
)
fig2.show()

## 8. Customer Segmentation — RFM Analysis
> **RFM** = Recency, Frequency, Monetary Value  
> A classic marketing technique to segment customers by behaviour

In [13]:
# Set reference date (day after last transaction)
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate RFM
rfm = df.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary  = ('Revenue', 'sum')
).reset_index()

print(rfm.describe())
rfm.head()

        Customer ID      Recency    Frequency       Monetary
count   5878.000000  5878.000000  5878.000000    5878.000000
mean   15315.313542   201.331916     6.289384    3018.616737
std     1715.572666   209.338707    13.009406   14737.731040
min    12346.000000     1.000000     1.000000       2.950000
25%    13833.250000    26.000000     1.000000     348.762500
50%    15314.500000    96.000000     3.000000     898.915000
75%    16797.750000   380.000000     7.000000    2307.090000
max    18287.000000   739.000000   398.000000  608821.650000


,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [14]:
# Score each dimension 1-4
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1,2,3,4])

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Segment customers
def segment_customer(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])
    score = r + f + m
    if score >= 10:
        return ' Champions'
    elif score >= 8:
        return ' Loyal Customers'
    elif score >= 6:
        return ' Potential Loyalists'
    elif r >= 3:
        return 'New Customers'
    elif score >= 4:
        return ' At Risk'
    else:
        return 'Lost Customers'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Plot segments
seg_counts = rfm['Segment'].value_counts().reset_index()
fig = px.pie(
    seg_counts,
    names='Segment',
    values='count',
    title='Customer Segments (RFM Analysis)',
    hole=0.4
)
fig.show()
print(rfm['Segment'].value_counts())

Segment
 Champions              1738
 Potential Loyalists    1220
 Loyal Customers        1190
 At Risk                1017
Lost Customers           575
New Customers            138
Name: count, dtype: int64


## 9. Revenue Forecasting (Simple Moving Average)

In [18]:
# Monthly revenue for forecasting
monthly_rev = df.groupby(df['InvoiceDate'].dt.to_period('M'))['Revenue'].sum()
monthly_rev.index = monthly_rev.index.to_timestamp()

# 3-month moving average
monthly_rev_df = monthly_rev.reset_index()
monthly_rev_df.columns = ['Month', 'Revenue']
monthly_rev_df['MA_3'] = monthly_rev_df['Revenue'].rolling(window=3).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly_rev_df['Month'], y=monthly_rev_df['Revenue'],
    name='Actual Revenue', line=dict(color='#2196F3')
))
fig.add_trace(go.Scatter(
    x=monthly_rev_df['Month'], y=monthly_rev_df['MA_3'],
    name='3-Month Moving Average', line=dict(color='#FF5722', dash='dash')
))
fig.update_layout(
    title=' Revenue Forecast — 3-Month Moving Average',
    xaxis_title='Month',
    yaxis_title='Revenue (£)'
)
fig.show()

## 10.  Key Business Insights

In [19]:
print('=' * 55)
print('         UK RETAIL SALES — KEY INSIGHTS')
print('=' * 55)

print(f'\n Total Revenue         : £{df["Revenue"].sum():>12,.2f}')
print(f'Total Orders          : {df["Invoice"].nunique():>12,}')
print(f' Unique Customers      : {df["Customer ID"].nunique():>12,}')
print(f' Unique Products       : {df["StockCode"].nunique():>12,}')
print(f' Countries Served      : {df["Country"].nunique():>12}')

avg_order = df.groupby('Invoice')['Revenue'].sum().mean()
print(f'Avg Order Value       : £{avg_order:>12,.2f}')

best_month = monthly_rev_df.loc[monthly_rev_df['Revenue'].idxmax()]
print(f' Best Month            : {str(best_month["Month"])[:7]} — £{best_month["Revenue"]:,.2f}')

top_country = country_rev.iloc[0]
print(f'Top Country           : {top_country["Country"]} — £{top_country["Revenue"]:,.2f}')

top_product = top_products.iloc[0]
print(f' Top Product           : {top_product["Description"][:30]}')

print('\n Recommendations:')
print('  1. Focus marketing spend in Nov-Dec (peak season)')
print('  2. Champions segment = highest LTV — reward them')
print('  3. Re-engage At Risk customers with targeted offers')
print('  4. Expand into top export markets outside UK')
print('=' * 55)

         UK RETAIL SALES — KEY INSIGHTS

 Total Revenue         : £17,743,429.18
Total Orders          :       36,969
 Unique Customers      :        5,878
 Unique Products       :        4,631
 Countries Served      :           41
Avg Order Value       : £      479.95
 Best Month            : 2010-11 — £1,172,336.04
Top Country           : United Kingdom — £14,723,147.52
 Top Product           : REGENCY CAKESTAND 3 TIER

 Recommendations:
  1. Focus marketing spend in Nov-Dec (peak season)
  2. Champions segment = highest LTV — reward them
  3. Re-engage At Risk customers with targeted offers
  4. Expand into top export markets outside UK


---
##  Summary

In this project we:
- Cleaned and processed **real UK retail transaction data**
- Analysed **sales trends** by month, day, and hour
- Identified **top products** by revenue and quantity
- Mapped **global sales distribution**
- Segmented customers using **RFM Analysis**
- Built a simple **revenue forecast** using moving averages

**Tech Stack:** Python | Pandas | NumPy | Plotly | Seaborn | Matplotlib

**Author:** Kiruthik Raam D V | MSc Data Science, University of Essex  
**GitHub:** [github.com/KiruthikRaamDV](https://github.com/KiruthikRaamDV)